In [1]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())

%load_ext autoreload
%autoreload 2

from src.config import Configuration
CONFIG = Configuration()

/home/turbotowerlnx/Documents/Master/ALC/ALC-Lab-Final/notebooks
/home/turbotowerlnx/Documents/Master/ALC/ALC-Lab-Final/app


# Extract embeddings

In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# 1. Load Model (Using 2B version to save VRAM. Use 7B if you have 24GB+ VRAM)
model_id = "Qwen/Qwen2-VL-2B-Instruct"
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id, 
    device_map="cuda", 
    torch_dtype=torch.float16
).eval()

processor = AutoProcessor.from_pretrained(model_id)

def get_video_embedding(video_path):
    # 2. Format input. Setting fps=1 mimics our previous 1-frame-per-second logic to save VRAM.
    messages = [
        {"role": "user", "content": [
            {"type": "video", "video": video_path, "max_pixels": 360 * 360, "fps": 1.0},
        ]}
    ]
    
    # 3. Process video into PyTorch tensors
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[""], # Empty text, we only want vision features
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to("cuda")

    # 4. Extract purely visual embeddings
    with torch.no_grad():
        # Access it through model.model
        visual_features = model.model.visual( # or model.model.vision_tower(
            inputs.pixel_values_videos, 
            grid_thw=inputs.video_grid_thw
        )
        # 5. Mean pool: Average all tokens to get ONE final vector for the video
        video_embedding = visual_features.last_hidden_state.mean(dim=0)

    return video_embedding.cpu().numpy()

# Usage:
# video_path = os.path.join(CONFIG.videos_path, "6630477888766348549.mp4")
video_path = os.path.join(CONFIG.videos_path, "7298026909340814625.mp4")
vector = get_video_embedding(video_path)
print(vector.shape) # Output: A 1D array, e.g., (1536,)

Loading weights: 100%|██████████| 729/729 [00:00<00:00, 1769.06it/s]


(1280,)


In [3]:
vector

array([2.562 , 0.4783, 2.898 , ..., 2.822 , 1.783 , 1.27  ],
      shape=(1280,), dtype=float16)

In [5]:
vector

array([3.273, 1.634, 2.576, ..., 4.285, 1.75 , 3.324],
      shape=(1280,), dtype=float16)